In [ ]:
# ============================================================
# NOTEBOOK 09 — TEMPORAL GENERALIZATION / DISTRIBUTION-SHIFT ANALYSIS
# ANALYSIS ONLY — NO RETRAINING / NO MODEL SELECTION
#
# Uses the frozen 07D common supervised samples.
# Quantifies Train / Validation / Test differences for:
#   T, RH, CO2 at prediction origin t
#   |ΔT15| = |T(t+15)-T(t)|
#
# Outputs:
# - descriptive statistics
# - standardized mean differences (SMD)
# - Wasserstein distance (SciPy if available)
# - Kolmogorov-Smirnov statistics (SciPy if available)
# - Jensen-Shannon divergence from common histograms
# - empirical PSI (diagnostic)
# - per-device comparisons
# - regime proportions for |ΔT15|
# - figures
#
# IMPORTANT INTERPRETATION:
# Statistical significance alone does NOT establish a practically important
# distribution shift. Effect magnitude + distributional distance + temporal
# dynamics should be interpreted together.
# ============================================================

from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
IN = ROOT / "outputs" / "07D_final_alignment"
OUT = ROOT / "outputs" / "09_temporal_generalization_shift"
OUT.mkdir(parents=True, exist_ok=True)

SRC = IN / "02_tabular_core_15_unscaled.csv"
assert SRC.exists(), f"Missing: {SRC}"

df = pd.read_csv(SRC)
print("="*88)
print("NOTEBOOK 09 — TEMPORAL GENERALIZATION / DISTRIBUTION-SHIFT ANALYSIS")
print("="*88)
print("Input :", SRC)
print("Output:", OUT)
print("Rows  :", len(df))
print("Columns:", len(df.columns))

# ---------- identify required columns ----------
TARGET_COL = "target_temperature_t_plus_15"
required = ["uuid","split",TARGET_COL]
for c in required:
    assert c in df.columns, f"Missing required column: {c}"

# Frozen 07D naming
candidates = {
    "T_t": ["temperature_t"],
    "RH_t": ["relative_humidity_t","humidity_t","RH_t"],
    "CO2_t": ["co2_t","CO2_t"],
}
resolved={}
for label,names in candidates.items():
    found=[c for c in names if c in df.columns]
    assert len(found)==1, f"Could not uniquely resolve {label}; candidates={names}; columns={df.columns.tolist()}"
    resolved[label]=found[0]

time_col = "target_timestamp" if "target_timestamp" in df.columns else (
    "timestamp" if "timestamp" in df.columns else None
)
assert time_col is not None, "No timestamp/target_timestamp column found."
df[time_col]=pd.to_datetime(df[time_col],utc=True)

split_map={}
for s in df["split"].astype(str).unique():
    sl=s.lower()
    if sl=="train": split_map[s]="train"
    elif sl in ["validation","val"]: split_map[s]="validation"
    elif sl=="test": split_map[s]="test"
df["split_std"]=df["split"].map(split_map)
assert df["split_std"].notna().all()

expected={"train":64701,"validation":13825,"test":13790}
actual=df["split_std"].value_counts().to_dict()
for s,n in expected.items():
    assert actual.get(s)==n, (s,actual.get(s),n)

# Current-state variables and target dynamics
work=pd.DataFrame({
    "uuid":df["uuid"].astype(str),
    "split":df["split_std"],
    "time":df[time_col],
    "T_t":pd.to_numeric(df[resolved["T_t"]],errors="coerce"),
    "RH_t":pd.to_numeric(df[resolved["RH_t"]],errors="coerce"),
    "CO2_t":pd.to_numeric(df[resolved["CO2_t"]],errors="coerce"),
    "T_target":pd.to_numeric(df[TARGET_COL],errors="coerce"),
})
work["delta_T15"]=work["T_target"]-work["T_t"]
work["abs_delta_T15"]=work["delta_T15"].abs()
assert work.notna().all().all()

variables=["T_t","RH_t","CO2_t","delta_T15","abs_delta_T15"]

# ---------- descriptive ----------
desc=[]
for split in ["train","validation","test"]:
    d=work[work.split==split]
    for var in variables:
        x=d[var].to_numpy()
        desc.append({
            "split":split,"variable":var,"n":len(x),
            "mean":np.mean(x),"SD":np.std(x,ddof=1),
            "median":np.median(x),
            "p05":np.quantile(x,.05),"p25":np.quantile(x,.25),
            "p75":np.quantile(x,.75),"p95":np.quantile(x,.95),
            "min":np.min(x),"max":np.max(x)
        })
desc=pd.DataFrame(desc)
desc.to_csv(OUT/"01_descriptive_statistics.csv",index=False)

# ---------- distribution metrics ----------
try:
    from scipy.stats import ks_2samp, wasserstein_distance
    HAVE_SCIPY=True
except Exception:
    HAVE_SCIPY=False
    print("NOTE: SciPy unavailable: KS and Wasserstein columns will be NaN.")

def smd(a,b):
    va=np.var(a,ddof=1); vb=np.var(b,ddof=1)
    pooled=np.sqrt((va+vb)/2)
    return float((np.mean(b)-np.mean(a))/pooled) if pooled>0 else np.nan

def js_divergence(a,b,bins=60):
    lo=min(np.min(a),np.min(b)); hi=max(np.max(a),np.max(b))
    if hi<=lo: return 0.0
    edges=np.linspace(lo,hi,bins+1)
    pa=np.histogram(a,bins=edges)[0].astype(float)
    pb=np.histogram(b,bins=edges)[0].astype(float)
    eps=1e-12
    pa=(pa+eps)/(pa.sum()+eps*len(pa))
    pb=(pb+eps)/(pb.sum()+eps*len(pb))
    m=.5*(pa+pb)
    return float(.5*np.sum(pa*np.log(pa/m))+.5*np.sum(pb*np.log(pb/m)))

def psi_train_bins(train,other,bins=10):
    # quantile bins defined ONLY from train; diagnostic, not a hypothesis test.
    edges=np.unique(np.quantile(train,np.linspace(0,1,bins+1)))
    if len(edges)<3: return np.nan
    edges[0]=-np.inf; edges[-1]=np.inf
    pa=np.histogram(train,bins=edges)[0].astype(float)
    pb=np.histogram(other,bins=edges)[0].astype(float)
    eps=1e-6
    pa=np.maximum(pa/pa.sum(),eps); pb=np.maximum(pb/pb.sum(),eps)
    return float(np.sum((pb-pa)*np.log(pb/pa)))

pairs=[("train","validation"),("train","test"),("validation","test")]
dist=[]
for var in variables:
    for a_name,b_name in pairs:
        a=work.loc[work.split==a_name,var].to_numpy()
        b=work.loc[work.split==b_name,var].to_numpy()
        row={
            "variable":var,"reference":a_name,"comparison":b_name,
            "mean_reference":np.mean(a),"mean_comparison":np.mean(b),
            "mean_difference":np.mean(b)-np.mean(a),
            "SMD":smd(a,b),
            "SMD_abs":abs(smd(a,b)),
            "Jensen_Shannon_divergence":js_divergence(a,b),
            "PSI_train_defined_bins":psi_train_bins(a,b) if a_name=="train" else np.nan
        }
        if HAVE_SCIPY:
            ks=ks_2samp(a,b,alternative="two-sided",method="auto")
            row["KS_statistic"]=float(ks.statistic)
            row["KS_pvalue"]=float(ks.pvalue)
            row["Wasserstein_distance"]=float(wasserstein_distance(a,b))
        else:
            row["KS_statistic"]=row["KS_pvalue"]=row["Wasserstein_distance"]=np.nan
        dist.append(row)
dist=pd.DataFrame(dist)
dist.to_csv(OUT/"02_distribution_distance_metrics.csv",index=False)

# ---------- per-device train vs test ----------
pdev=[]
for uuid in sorted(work.uuid.unique()):
    du=work[work.uuid==uuid]
    for var in variables:
        a=du.loc[du.split=="train",var].to_numpy()
        b=du.loc[du.split=="test",var].to_numpy()
        row={"uuid":uuid,"variable":var,
             "train_mean":np.mean(a),"test_mean":np.mean(b),
             "mean_difference":np.mean(b)-np.mean(a),
             "SMD":smd(a,b),"SMD_abs":abs(smd(a,b)),
             "Jensen_Shannon_divergence":js_divergence(a,b)}
        if HAVE_SCIPY:
            ks=ks_2samp(a,b)
            row["KS_statistic"]=float(ks.statistic)
            row["KS_pvalue"]=float(ks.pvalue)
            row["Wasserstein_distance"]=float(wasserstein_distance(a,b))
        pdev.append(row)
pdev=pd.DataFrame(pdev)
pdev.to_csv(OUT/"03_per_device_train_test_shift.csv",index=False)

# ---------- |ΔT15| regimes ----------
def regime(x):
    if x < .02: return "<0.02"
    if x < .05: return "0.02-0.05"
    if x < .10: return "0.05-0.10"
    if x < .20: return "0.10-0.20"
    return ">=0.20"

work["change_regime"]=work["abs_delta_T15"].map(regime)
order=["<0.02","0.02-0.05","0.05-0.10","0.10-0.20",">=0.20"]
reg=(work.groupby(["split","change_regime"]).size().rename("n").reset_index())
reg["pct"]=reg.groupby("split")["n"].transform(lambda x:100*x/x.sum())
reg["change_regime"]=pd.Categorical(reg["change_regime"],categories=order,ordered=True)
reg=reg.sort_values(["split","change_regime"])
reg.to_csv(OUT/"04_deltaT15_regime_proportions.csv",index=False)

# ---------- time coverage ----------
coverage=(work.groupby("split")
          .agg(n=("T_t","size"),devices=("uuid","nunique"),
               start=("time","min"),end=("time","max"),
               unique_times=("time","nunique"))
          .reset_index())
coverage.to_csv(OUT/"05_split_time_coverage.csv",index=False)

# ---------- figures ----------
plt.figure(figsize=(8,5))
for s in ["train","validation","test"]:
    x=work.loc[work.split==s,"T_t"]
    plt.hist(x,bins=50,density=True,histtype="step",linewidth=1.5,label=s)
plt.xlabel("Indoor temperature at t (°C)")
plt.ylabel("Density")
plt.title("Indoor temperature distributions across temporal splits")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"Fig09_1_temperature_distributions.png",dpi=300)
plt.close()

plt.figure(figsize=(8,5))
for s in ["train","validation","test"]:
    x=work.loc[work.split==s,"abs_delta_T15"]
    # cap visual axis at p99.5 overall only for readability; data analysis is uncapped
    cap=work["abs_delta_T15"].quantile(.995)
    plt.hist(x[x<=cap],bins=50,density=True,histtype="step",linewidth=1.5,label=s)
plt.xlabel("|ΔT15| (°C)")
plt.ylabel("Density")
plt.title("15-min temperature-change distributions across temporal splits")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"Fig09_2_deltaT15_distributions.png",dpi=300)
plt.close()

pivot=reg.pivot(index="change_regime",columns="split",values="pct").reindex(order)
pivot.plot(kind="bar",figsize=(9,5))
plt.ylabel("Share of samples (%)")
plt.xlabel("|ΔT15| regime (°C)")
plt.title("Temperature-dynamics regimes across temporal splits")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT/"Fig09_3_deltaT15_regime_proportions.png",dpi=300)
plt.close()

# ---------- decision-support summary ----------
tt=dist[(dist.reference=="train")&(dist.comparison=="test")].copy()
tt.to_csv(OUT/"06_train_test_shift_summary.csv",index=False)

manifest={
    "analysis_only":True,
    "no_retraining":True,
    "source":"07D frozen primary tabular unscaled common samples",
    "counts":expected,
    "variables_analyzed":variables,
    "distribution_metrics":["SMD","Jensen-Shannon divergence","PSI diagnostic",
                            "KS statistic/p-value if SciPy available",
                            "Wasserstein distance if SciPy available"],
    "interpretation_rule":"Do not label distribution shift solely from p-values; interpret magnitude, distribution distances, per-device consistency, and |ΔT15| regime changes.",
    "scenario_naming_before_analysis":"forward temporal generalization"
}
with open(OUT/"07_analysis_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)

print("\nSPLIT COVERAGE")
print(coverage.to_string(index=False))

print("\nTRAIN → TEST DISTRIBUTION SUMMARY")
cols=["variable","mean_reference","mean_comparison","mean_difference","SMD","SMD_abs",
      "KS_statistic","KS_pvalue","Wasserstein_distance",
      "Jensen_Shannon_divergence","PSI_train_defined_bins"]
print(tt[cols].to_string(index=False))

print("\n|ΔT15| REGIME PROPORTIONS (%)")
print(reg[["split","change_regime","n","pct"]].to_string(index=False))

print("\nVALIDATION")
print("✓ Frozen 07D samples used")
print("✓ No model training or model selection")
print("✓ Train / Validation / Test analyzed chronologically")
print("✓ T, RH, CO2 and 15-min temperature dynamics quantified")
print("✓ Per-device Train→Test analysis saved")
print("✓ Distribution distances + effect sizes saved")
print("✓ Figures saved at 300 dpi")
print("\nIMPORTANT: Keep the label 'forward temporal generalization' until these")
print("magnitudes are interpreted; KS p-values alone are not evidence of a")
print("practically important distribution shift.")


In [ ]:
from pathlib import Path
import pandas as pd

path = Path.cwd() / "outputs" / "07D_final_alignment" / "02_tabular_core_15_unscaled.csv"

df = pd.read_csv(path, nrows=5)

print("=== COLONNES ===")
for i, col in enumerate(df.columns):
    print(i, ":", col)

print("\n=== PREMIERES LIGNES ===")
print(df.to_string(index=False))